Install the libraries

In [54]:
!pip install scikit-learn
!pip install torch


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
import os

# ---------------- CONFIG ----------------
INPUT_FILE = "datasets/clean_eye_data.npz"
OUTPUT_FILE = "datasets/clean_eye_data_lstm.npz"
SEQ_LEN = 5   # number of windows per sequence
# ----------------------------------------


def build_lstm_sequences(X, y, seq_len):
    """
    X: (N, C, T)
    y: (N,)
    Returns:
        X_seq: (N_seq, seq_len, C*T)
        y_seq: (N_seq,)
    """
    N, C, T = X.shape

    X_seq = []
    y_seq = []

    for i in range(N - seq_len + 1):
        seq = X[i:i + seq_len]              # (seq_len, C, T)
        seq_flat = seq.reshape(seq_len, -1) # (seq_len, C*T)

        X_seq.append(seq_flat)
        y_seq.append(y[i + seq_len - 1])    # label of last window

    return np.array(X_seq), np.array(y_seq)


# ---------------- LOAD CLEAN DATA ----------------
data = np.load(INPUT_FILE)

X_clean = data["X"]   # (N, 8, 250)
y_clean = data["y"]   # (N,)

y_clean = y_clean - 1

print("Loaded clean data:")
print("X:", X_clean.shape)
print("y:", y_clean.shape)
print("Label distribution:", np.unique(y_clean, return_counts=True))


# ---------------- BUILD SEQUENCES ----------------
X_seq, y_seq = build_lstm_sequences(X_clean, y_clean, SEQ_LEN)

print("\nLSTM-ready data:")
print("X_seq:", X_seq.shape)
print("y_seq:", y_seq.shape)
print("Sequence label distribution:", np.unique(y_seq, return_counts=True))


# ---------------- SAVE ----------------
np.savez(
    OUTPUT_FILE,
    X=X_seq,
    y=y_seq,
    seq_len=SEQ_LEN,
    feature_dim=X_seq.shape[2]
)

print(f"\nSaved LSTM dataset → {OUTPUT_FILE}")

Loaded clean data:
X: (400, 8, 250)
y: (400,)
Label distribution: (array([0, 1]), array([200, 200]))

LSTM-ready data:
X_seq: (396, 5, 2000)
y_seq: (396,)
Sequence label distribution: (array([0, 1]), array([196, 200]))

Saved LSTM dataset → datasets/clean_eye_data_lstm.npz


In [56]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_seq,
    y_seq,
    test_size=0.2,
    stratify=y_seq,
    random_state=42
)

Training The Neural Network, Set Up Data

In [57]:
import torch
from torch.utils.data import Dataset, DataLoader

class EEGSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = EEGSequenceDataset(X_train, y_train)
val_ds   = EEGSequenceDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=False)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)

Model Architecture

In [58]:
import torch.nn as nn

class EyeStateLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]     
        logits = self.fc(last)
        return logits

Training Loop Log Loss every epoch for train and test set

In [59]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = EyeStateLSTM(
    input_dim=X_seq.shape[2],
    hidden_dim=64
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


Using device: cuda


In [60]:
peak_all_time_test_acc = 0.0

In [61]:
num_epochs = 100
peak_test_acc = 0.0
peak_test_acc_epoch = 0
record_broken = False

for epoch in range(num_epochs):

    # ---------------- TRAIN ----------------
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)   # move to GPU
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)       # logits (B, 2)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * y_batch.size(0)

        preds = torch.argmax(outputs, dim=1)
        train_correct += (preds == y_batch).sum().item()
        train_total += y_batch.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # ---------------- TEST ----------------
    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)   # move to GPU
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            test_loss += loss.item() * y_batch.size(0)

            preds = torch.argmax(outputs, dim=1)
            test_correct += (preds == y_batch).sum().item()
            test_total += y_batch.size(0)

    test_loss /= test_total
    test_acc = test_correct / test_total

    # ---------------- CHECKPOINTS ----------------
    if test_acc > peak_test_acc:
        peak_test_acc = test_acc
        peak_test_acc_epoch = epoch + 1

    if test_acc > peak_all_time_test_acc:
        peak_all_time_test_acc = test_acc
        torch.save({
            "model_state": model.state_dict(),
            "epoch": epoch + 1,
            "test_acc": test_acc
        }, "assets/eye_model.pth")

        print(
            f" New BEST model saved | "
            f"Test Acc: {test_acc*100:.2f}% | Epoch {epoch+1}"
        )
        record_broken = True

    # ---------------- LOG ----------------
    print(
        f"Epoch {epoch+1:03d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% | "
        f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc*100:.2f}%"
    )

print(f"\nPeak Test Accuracy: {peak_test_acc*100:.2f}%")
print(f"Peak Accuracy Epoch: {peak_test_acc_epoch}")

if record_broken:
    print(
        f" All-time best test accuracy: "
        f"{peak_all_time_test_acc*100:.2f}% "
        f"(epoch {peak_test_acc_epoch})")

 New BEST model saved | Test Acc: 61.25% | Epoch 1
Epoch 001/100 | Train Loss: 0.7137, Train Acc: 47.47% | Test Loss: 0.6955, Test Acc: 61.25%
Epoch 002/100 | Train Loss: 0.6836, Train Acc: 61.08% | Test Loss: 0.6731, Test Acc: 56.25%
 New BEST model saved | Test Acc: 76.25% | Epoch 3
Epoch 003/100 | Train Loss: 0.6613, Train Acc: 65.51% | Test Loss: 0.6511, Test Acc: 76.25%
 New BEST model saved | Test Acc: 78.75% | Epoch 4
Epoch 004/100 | Train Loss: 0.6402, Train Acc: 75.63% | Test Loss: 0.6301, Test Acc: 78.75%
 New BEST model saved | Test Acc: 88.75% | Epoch 5
Epoch 005/100 | Train Loss: 0.6200, Train Acc: 89.56% | Test Loss: 0.6102, Test Acc: 88.75%
Epoch 006/100 | Train Loss: 0.6008, Train Acc: 90.51% | Test Loss: 0.5913, Test Acc: 88.75%
Epoch 007/100 | Train Loss: 0.5824, Train Acc: 91.14% | Test Loss: 0.5734, Test Acc: 85.00%
Epoch 008/100 | Train Loss: 0.5649, Train Acc: 90.82% | Test Loss: 0.5563, Test Acc: 85.00%
Epoch 009/100 | Train Loss: 0.5483, Train Acc: 91.77% | Test

Metrics

In [62]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)    # move to GPU
        y_batch = y_batch.to(device)

        outputs = model(X_batch)        # logits (B, 2)
        preds = torch.argmax(outputs, dim=1)

        all_preds.append(preds.cpu().numpy())
        all_labels.append(y_batch.cpu().numpy())

# Concatenate batches
y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

# Classification report
print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Eyes Open", "Eyes Closed"]
    )
)

Confusion Matrix:
[[33  7]
 [ 1 39]]

Classification Report:
              precision    recall  f1-score   support

   Eyes Open       0.97      0.82      0.89        40
 Eyes Closed       0.85      0.97      0.91        40

    accuracy                           0.90        80
   macro avg       0.91      0.90      0.90        80
weighted avg       0.91      0.90      0.90        80

